# 10 - KPIs oficiales del dashboard VcM: real vs sintetico

Visualiza **todos los KPIs solicitados por la contraparte de VcM**, comparando en
cada uno lo que puede mostrar el **dataset real** (`data/clean/dashboard_real`)
contra el **sintetico** (`data/synthetic/dashboard_sintetico`). Ambos comparten
el mismo esquema de 39 columnas (notebook 08).

**Como leer cada seccion:**

- Cada KPI tiene su explicacion, y un grafico con **ambos datasets lado a lado**:
  izquierda el real, derecha el sintetico.
- Cuando el real no tiene el dato (columna estructuralmente vacia), el panel
  izquierdo lo dice de forma explicita: **"SIN DATOS EN FUENTE REAL"**, en vez de
  un grafico vacio silencioso.
- Todo panel sintetico lleva la etiqueta **DATOS SINTETICOS / DEMOSTRATIVOS**:
  esas cifras no son reales, son el molde visual del dashboard con captura
  completa.
- Azul = real, verde aqua = sintetico (igual que el notebook 09). En graficos con
  varias series (facultades, roles, sellos) cada serie conserva siempre el mismo
  color en ambos paneles.

Al cierre hay una **tabla resumen del estado de reporteria**: que KPIs son
calculables hoy con datos reales, cuales parcialmente y cuales no, y desde que
anio.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

CLEAN = Path("..") / "data" / "clean"
SYNTH = Path("..") / "data" / "synthetic"

real = pd.read_parquet(CLEAN / "dashboard_real.parquet")
sint = pd.read_parquet(SYNTH / "dashboard_sintetico.parquet")
assert list(real.columns) == list(sint.columns), "Esquemas distintos"
assert len(real) == len(sint)
print(f"OK: ambos datasets cargados con esquema identico ({len(real.columns)} columnas, {len(real)} filas)")

# ---- paleta (misma convencion del notebook 09) ----
C_REAL, C_SINT = "#2a78d6", "#1baf7a"
PAL8 = ["#2a78d6", "#1baf7a", "#eda100", "#008300", "#4a3aa7", "#e34948",
        "#e87ba4", "#eb6834"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
SEQ_AZUL = LinearSegmentedColormap.from_list("seq_azul", [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
ETIQ = "DATOS SINTETICOS / DEMOSTRATIVOS"

FACULTADES = sorted(set(real["facultad"].dropna()))
FCOLOR = {f: PAL8[i] for i, f in enumerate(FACULTADES)}      # color fijo por facultad
ROLES = ["charlista", "expositor", "asistente"]
RCOLOR = dict(zip(ROLES, PAL8[:3]))                          # color fijo por rol
SELLOS = [("sello_tecnologia", "Tecnologia"), ("sello_responsabilidad_social", "Resp. social"),
          ("sello_sustentabilidad", "Sustentabilidad"), ("sello_genero", "Genero")]
SCOLOR = {s: PAL8[i] for i, (s, _) in enumerate(SELLOS)}     # color fijo por sello

ANIOS = [2022, 2023, 2024, 2025]


def estilo(ax, titulo=None, eje="y"):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.grid(axis=eje, color=GRID, linewidth=0.8)
    ax.grid(axis="x" if eje == "y" else "y", visible=False)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, labelsize=9)
    if titulo:
        ax.set_title(titulo, color=INK, fontsize=10, loc="left")


def par(suptitulo, figsize=(12.5, 4)):
    """Figura 1x2: panel real (izq) y sintetico (der) con titulos fijos."""
    fig, (axr, axs) = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(suptitulo, color=INK, fontsize=12, x=0.02, ha="left")
    return fig, axr, axs


def sin_datos(ax, detalle):
    """Panel explicito cuando la fuente real no captura el dato."""
    ax.axis("off")
    ax.text(0.5, 0.56, "SIN DATOS EN FUENTE REAL", ha="center", va="center",
            color=MUTED, fontsize=12, weight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.40, detalle, ha="center", va="center", color=MUTED,
            fontsize=9, style="italic", transform=ax.transAxes, wrap=True)


def es_realizada(v):
    """Clasifica estado_sisav: realizada (finalizada o en ejecucion) vs no."""
    if pd.isna(v):
        return np.nan
    nv = str(v).strip().lower()
    if nv.startswith("no "):
        return False
    return ("finaliza" in nv) or ("ejecu" in nv)


def barras_anio(ax, serie, color, titulo, pct=False):
    vals = serie.reindex(ANIOS)
    ax.bar([str(a) for a in ANIOS], vals.values, width=0.55, color=color,
           edgecolor="white", linewidth=0.5)
    for i, v in enumerate(vals.values):
        if not pd.isna(v):
            ax.text(i, v, f"{v:.0f}" + ("%" if pct else ""), ha="center",
                    va="bottom", color=MUTED, fontsize=9)
    estilo(ax, titulo)


def apilado_facultad(ax, df, col_valor, titulo, agg="sum"):
    """Barras por anio apiladas por facultad, colores fijos por facultad."""
    if agg == "sum":
        tab = df.groupby(["anio", "facultad"])[col_valor].sum(min_count=1).unstack()
    else:
        tab = df.groupby(["anio", "facultad"]).size().unstack()
    tab = tab.reindex(index=ANIOS, columns=FACULTADES)
    base = np.zeros(len(ANIOS))
    x = np.arange(len(ANIOS))
    for f in FACULTADES:
        vals = tab[f].fillna(0).to_numpy(dtype=float)
        ax.bar(x, vals, bottom=base, width=0.55, color=FCOLOR[f],
               edgecolor="white", linewidth=1.2, label=f)
        base += vals
    ax.set_xticks(x, [str(a) for a in ANIOS])
    estilo(ax, titulo)


pd.set_option("display.width", 220)

## KPI 1 - % de dominios disciplinares que contribuyen al proceso formativo

Porcentaje de iniciativas que declaran dominios disciplinares
(`dominios_disciplinares`, multivalor), por anio. En el real el dato **existe
solo desde 2024** (cobertura ~88%): los anios 2022-2023 aparecen sin barra
porque el concepto no existia en la fuente (el area generica de esos anios es
otra cosa). Ademas se muestra el numero de dominios distintos declarados por
facultad.

Limite honesto: el KPI literal ("% de los dominios del plan de estudios de cada
carrera cubiertos por VcM") requiere el **catalogo oficial de dominios por
carrera**, que no esta en los datos. Lo que se puede medir con la fuente es la
declaracion de dominios en las iniciativas; si VcM entrega el catalogo, el
calculo literal es directo sobre esta misma columna.

In [ ]:
fig, axr, axs = par("KPI 1: % de iniciativas que declaran dominios disciplinares, por anio")
for ax, df, color, tit in [(axr, real, C_REAL, "Dataset REAL (dato desde 2024)"),
                           (axs, sint, C_SINT, ETIQ)]:
    grp = df.groupby("anio")["dominios_disciplinares"]
    pct = grp.apply(lambda s: 100 * s.notna().mean() if s.notna().any() else np.nan)
    barras_anio(ax, pct, color, tit, pct=True)
    ax.set_ylim(0, 115)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

In [ ]:
# dominios distintos declarados por facultad (riqueza disciplinar de la VcM)
def dominios_distintos(df):
    out = {}
    for f, sub in df.groupby("facultad")["dominios_disciplinares"]:
        toks = set()
        for v in sub.dropna():
            toks.update(t.strip() for t in str(v).split(";") if t.strip())
        out[f] = len(toks)
    return pd.Series(out).reindex(FACULTADES).fillna(0)

fig, axr, axs = par("KPI 1 (complemento): dominios disciplinares distintos por facultad")
for ax, df, tit in [(axr, real, "Dataset REAL (2024-2025)"), (axs, sint, ETIQ)]:
    cnt = dominios_distintos(df)
    ax.bar(FACULTADES, cnt.values, width=0.55,
           color=[FCOLOR[f] for f in FACULTADES], edgecolor="white", linewidth=0.5)
    for i, v in enumerate(cnt.values):
        ax.text(i, v, f"{v:.0f}", ha="center", va="bottom", color=MUTED, fontsize=9)
    estilo(ax, tit)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 2 - % de iniciativas de VcM realizadas, por anio

Proporcion de iniciativas efectivamente realizadas (estado finalizada o en
ejecucion) sobre el total registrado en el anio. Se deriva de `estado_sisav`
normalizando el vocabulario heterogeneo de la fuente (FINALIZADO / Finalizada /
EJECUCION cuentan como realizadas; NO EJECUTADA / NO REALIZADO como no
realizadas; INGRESADAS queda fuera del numerador). **Calculable con datos
reales** en todo el periodo. El desglose por facultad usa los mismos colores
fijos en ambos paneles.

In [ ]:
fig, axr, axs = par("KPI 2: % de iniciativas realizadas por anio")
for ax, df, color, tit in [(axr, real, C_REAL, "Dataset REAL"),
                           (axs, sint, C_SINT, ETIQ)]:
    ok = df["estado_sisav"].map(es_realizada)
    pct = ok.groupby(df["anio"]).mean() * 100
    barras_anio(ax, pct, color, tit, pct=True)
    ax.set_ylim(0, 105)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 3 - % de iniciativas realizadas en comunas de la RM, por anio

Requiere `comuna_rm`, que la fuente real **no captura**: en el real esta 100%
vacia. En el sintetico esta poblada por diseno (toda iniciativa demo tiene una
comuna real de la RM), asi que su % es 100 en todos los anios: el valor del
panel es demostrar el KPI funcionando, no la cifra.

In [ ]:
fig, axr, axs = par("KPI 3: % de iniciativas en comunas de la RM por anio")
sin_datos(axr, "La fuente no captura la comuna de ejecucion\n(comuna_rm vacia en todo el periodo)")
pct = sint.groupby("anio")["comuna_rm"].apply(lambda s: 100 * s.notna().mean())
barras_anio(axs, pct, C_SINT, ETIQ, pct=True)
axs.set_ylim(0, 115)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 4 - Numero de iniciativas ejecutadas, total y por facultad

Conteo de iniciativas por facultad (suma de todo el periodo 2022-2025), con el
total institucional en el titulo. **Calculable con datos reales** en todo el
periodo. Las diferencias entre paneles son minimas porque el sintetico replica
las proporciones reales por construccion.

In [ ]:
fig, axr, axs = par("KPI 4: numero de iniciativas por facultad (2022-2025)")
for ax, df, tit in [(axr, real, "Dataset REAL"), (axs, sint, ETIQ)]:
    cnt = df["facultad"].value_counts().reindex(FACULTADES).fillna(0)
    ax.bar(cnt.index, cnt.values, width=0.55,
           color=[FCOLOR[f] for f in cnt.index], edgecolor="white", linewidth=0.5)
    for i, v in enumerate(cnt.values):
        ax.text(i, v, f"{v:.0f}", ha="center", va="bottom", color=MUTED, fontsize=9)
    estilo(ax, f"{tit}  (total: {int(df['facultad'].notna().sum())})")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 5 - Numero de iniciativas por instrumento y anio

Volumen anual por instrumento (VEDP, Extension, Titulados/VT, mas FCR, UTG y
Centralizadas segun el anio). **Calculable con datos reales** en todo el periodo.
Barras apiladas: cada instrumento conserva su color en ambos paneles.

In [ ]:
INSTRUMENTOS = sorted(set(real["instrumento"].dropna()))
ICOLOR = {i: PAL8[k] for k, i in enumerate(INSTRUMENTOS)}

fig, axr, axs = par("KPI 5: iniciativas por instrumento y anio", figsize=(12.5, 4.4))
for ax, df, tit in [(axr, real, "Dataset REAL"), (axs, sint, ETIQ)]:
    tab = df.groupby(["anio", "instrumento"]).size().unstack().reindex(
        index=ANIOS, columns=INSTRUMENTOS)
    x = np.arange(len(ANIOS))
    base = np.zeros(len(ANIOS))
    for ins in INSTRUMENTOS:
        vals = tab[ins].fillna(0).to_numpy(dtype=float)
        ax.bar(x, vals, bottom=base, width=0.55, color=ICOLOR[ins],
               edgecolor="white", linewidth=1.2, label=ins)
        base += vals
    ax.set_xticks(x, [str(a) for a in ANIOS])
    estilo(ax, tit)
axr.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=2)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 6 - Numero de estudiantes que participan

Suma de estudiantes participantes por anio, apilada por facultad. En el real el
dato **existe solo desde 2024** (2022-2023 no traen participantes desagregados:
los anios aparecen sin barra, no en cero). Las sumas ignoran NaN sin fabricar
ceros. El sintetico muestra la serie completa como se veria con captura total.

In [ ]:
fig, axr, axs = par("KPI 6: estudiantes participantes por anio y facultad", figsize=(12.5, 4.4))
apilado_facultad(axr, real, "n_estudiantes", "Dataset REAL (solo 2024-2025 tiene dato)")
apilado_facultad(axs, sint, "n_estudiantes", ETIQ)
axr.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=2)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 7 - Numero de academicos que realizan iniciativas

Misma logica que estudiantes: real **solo 2024-2025**, sintetico completo. Aca
se muestra el corte por instrumento (el corte por facultad es analogo al KPI 6).

In [ ]:
fig, axr, axs = par("KPI 7: academicos por anio e instrumento", figsize=(12.5, 4.4))
for ax, df, tit in [(axr, real, "Dataset REAL (solo 2024-2025 tiene dato)"), (axs, sint, ETIQ)]:
    tab = df.groupby(["anio", "instrumento"])["n_academicos"].sum(min_count=1).unstack()
    tab = tab.reindex(index=ANIOS, columns=INSTRUMENTOS)
    x = np.arange(len(ANIOS))
    base = np.zeros(len(ANIOS))
    for ins in INSTRUMENTOS:
        vals = tab[ins].fillna(0).to_numpy(dtype=float)
        ax.bar(x, vals, bottom=base, width=0.55, color=ICOLOR[ins],
               edgecolor="white", linewidth=1.2, label=ins)
        base += vals
    ax.set_xticks(x, [str(a) for a in ANIOS])
    estilo(ax, tit)
axr.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=2)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 8 - Numero de titulados que participan, por facultad y rol

El **total** de titulados por facultad es calculable con el real desde 2024. El
**desglose por rol** (charlista / expositor / asistente) solo existe en el
sintetico: la fuente real no registra roles.

In [ ]:
fig, axr, axs = par("KPI 8: titulados participantes por facultad", figsize=(12.5, 4.4))
tot = real.groupby("facultad")["n_titulados"].sum(min_count=1).reindex(FACULTADES)
axr.bar(FACULTADES, tot.fillna(0).values, width=0.55,
        color=[FCOLOR[f] for f in FACULTADES], edgecolor="white", linewidth=0.5)
estilo(axr, "Dataset REAL: total por facultad (dato 2024-2025)")
axr.tick_params(axis="x", rotation=45)

x = np.arange(len(FACULTADES))
base = np.zeros(len(FACULTADES))
for rol in ROLES:
    vals = sint.groupby("facultad")[f"n_titulados_{rol}"].sum().reindex(FACULTADES).fillna(0).to_numpy(dtype=float)
    axs.bar(x, vals, bottom=base, width=0.55, color=RCOLOR[rol],
            edgecolor="white", linewidth=1.2, label=rol.capitalize())
    base += vals
axs.set_xticks(x, FACULTADES, rotation=45)
estilo(axs, ETIQ + ": desglose por rol")
axs.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=3)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 9 - Numero de empleadores que participan, por facultad y rol

La fuente real **no tiene ninguna columna de empleadores** (confirmado en el
diagnostico del notebook 06): panel real sin datos. El sintetico demuestra el
KPI con el desglose por rol.

In [ ]:
fig, axr, axs = par("KPI 9: empleadores participantes por facultad y rol", figsize=(12.5, 4.4))
sin_datos(axr, "Ninguna planilla fuente 2022-2025 tiene columna de\nempleadores/emprendedores (diagnostico notebook 06)")
x = np.arange(len(FACULTADES))
base = np.zeros(len(FACULTADES))
for rol in ROLES:
    vals = sint.groupby("facultad")[f"n_empleadores_{rol}"].sum().reindex(FACULTADES).fillna(0).to_numpy(dtype=float)
    axs.bar(x, vals, bottom=base, width=0.55, color=RCOLOR[rol],
            edgecolor="white", linewidth=1.2, label=rol.capitalize())
    base += vals
axs.set_xticks(x, FACULTADES, rotation=45)
estilo(axs, ETIQ)
axs.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=3)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 10 - Numero de organizaciones de la sociedad civil, por facultad y rol

Igual que titulados: el **total** por facultad es real desde 2024
(`n_organizaciones_osc`); el **desglose por rol** solo existe en el sintetico.

In [ ]:
fig, axr, axs = par("KPI 10: organizaciones OSC participantes por facultad", figsize=(12.5, 4.4))
tot = real.groupby("facultad")["n_organizaciones_osc"].sum(min_count=1).reindex(FACULTADES)
axr.bar(FACULTADES, tot.fillna(0).values, width=0.55,
        color=[FCOLOR[f] for f in FACULTADES], edgecolor="white", linewidth=0.5)
estilo(axr, "Dataset REAL: total por facultad (dato 2024-2025)")
axr.tick_params(axis="x", rotation=45)

x = np.arange(len(FACULTADES))
base = np.zeros(len(FACULTADES))
for rol in ROLES:
    vals = sint.groupby("facultad")[f"n_organizaciones_osc_{rol}"].sum().reindex(FACULTADES).fillna(0).to_numpy(dtype=float)
    axs.bar(x, vals, bottom=base, width=0.55, color=RCOLOR[rol],
            edgecolor="white", linewidth=1.2, label=rol.capitalize())
    base += vals
axs.set_xticks(x, FACULTADES, rotation=45)
estilo(axs, ETIQ + ": desglose por rol")
axs.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=3)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 11 - Numero de competencias sello declaradas, desagregadas

Conteo de iniciativas que declaran cada competencia sello (tecnologico,
responsabilidad social, sustentabilidad, genero), por anio. Los flags se derivan
del texto real de `competencia_sello`, asi que este KPI **es calculable con
datos reales en todo el periodo** (una iniciativa puede declarar mas de un
sello, por eso las categorias no suman el total de iniciativas).

In [ ]:
fig, axr, axs = par("KPI 11: competencias sello declaradas por anio", figsize=(12.5, 4.4))
for ax, df, tit in [(axr, real, "Dataset REAL"), (axs, sint, ETIQ)]:
    x = np.arange(len(ANIOS))
    w = 0.19
    for k, (col, nombre) in enumerate(SELLOS):
        vals = df.groupby("anio")[col].sum().reindex(ANIOS).fillna(0).astype(float)
        ax.bar(x + (k - 1.5) * w, vals, width=w * 0.92, color=SCOLOR[col],
               edgecolor="white", linewidth=0.5, label=nombre)
    ax.set_xticks(x, [str(a) for a in ANIOS])
    estilo(ax, tit)
axr.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=2)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 12 - Numero de iniciativas que declaran internacionalizacion

La fuente real **no captura** internacionalizacion: panel real sin datos. En el
sintetico se parametrizo una tasa baja (~8% de las iniciativas) como referencia
visual del KPI.

In [ ]:
fig, axr, axs = par("KPI 12: iniciativas con internacionalizacion por anio")
sin_datos(axr, "La fuente no registra si la iniciativa tiene\ncomponente de internacionalizacion")
cnt = sint.groupby("anio")["internacionalizacion"].sum().astype(float)
barras_anio(axs, cnt, C_SINT, ETIQ)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 13 - Numero de catedras declaradas, por facultad

Iniciativas con catedra/asignatura asociada declarada. En el real el dato existe
**solo desde 2024 y con cobertura parcial** (~57%: en 2024 solo el instrumento
VEDP la trae; en 2025, 3 de 4 instrumentos). El conteo real es por eso un piso,
no el total de catedras asociadas.

In [ ]:
fig, axr, axs = par("KPI 13: iniciativas con catedra declarada por facultad")
for ax, df, tit in [(axr, real, "Dataset REAL (2024-2025, cobertura parcial ~57%)"),
                    (axs, sint, ETIQ)]:
    cnt = df[df["catedra_asignatura"].notna()]["facultad"].value_counts().reindex(FACULTADES).fillna(0)
    ax.bar(FACULTADES, cnt.values, width=0.55,
           color=[FCOLOR[f] for f in FACULTADES], edgecolor="white", linewidth=0.5)
    for i, v in enumerate(cnt.values):
        ax.text(i, v, f"{v:.0f}", ha="center", va="bottom", color=MUTED, fontsize=9)
    estilo(ax, tit)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 14 - Numero de iniciativas por ciclo de estudio

Ciclos 0 a 3 del modelo educativo. La fuente real (en las tablas limpias del
esquema v1) **no trae el ciclo**: panel real sin datos. El sintetico demuestra
la vista.

In [ ]:
fig, axr, axs = par("KPI 14: iniciativas por ciclo de estudio")
sin_datos(axr, "El ciclo del modelo educativo no esta en el esquema v1\n(existe parcialmente en la fuente; candidato a extraccion futura)")
ciclo = sint["ciclo_estudio"].value_counts().sort_index()
axs.bar([f"Ciclo {c}" for c in ciclo.index], ciclo.values.astype(float), width=0.55,
        color=C_SINT, edgecolor="white", linewidth=0.5)
for i, v in enumerate(ciclo.values):
    axs.text(i, v, str(int(v)), ha="center", va="bottom", color=MUTED, fontsize=9)
estilo(axs, ETIQ)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI 15 - % de iniciativas que cuentan con evidencia

Proporcion de iniciativas con evidencia de ejecucion (`evidencia` = SI), por
anio. **Calculable con datos reales en todo el periodo** (cobertura 96-100%):
en 2022-2023 se deriva de `Estado de Evidencia` (COMPLETO/INCOMPLETO cuentan
como SI, SIN EVIDENCIA como NO) y en 2024-2025 la fuente trae el campo SI/NO
directo. El % se calcula sobre las iniciativas con dato, ignorando NaN. En el
sintetico la proporcion de SI replica la real observada (~85%).

In [ ]:
fig, axr, axs = par("KPI 15: % de iniciativas con evidencia de ejecucion, por anio")
for ax, df, color, tit in [(axr, real, C_REAL, "Dataset REAL"),
                           (axs, sint, C_SINT, ETIQ)]:
    con_dato = df.dropna(subset=["evidencia"])
    pct = con_dato.groupby("anio")["evidencia"].apply(lambda s: 100 * (s == "SI").mean())
    barras_anio(ax, pct, color, tit, pct=True)
    ax.set_ylim(0, 115)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## KPI I19 (oficial de acreditacion) - Contribucion al plan de estudios

`I19 = actividades ejecutadas / actividades planificadas x 100`. **Calculable
con datos reales para 2022-2023**: la fuente capturo ambas cantidades en esos
anios (ejecutadas con cobertura 78-90%). Desde 2024 la fuente dejo de registrar
las ejecutadas, por eso el panel real no tiene barras en 2024-2025 (planificadas
si existe todos los anios, pero el cociente requiere ambas). El sintetico
muestra la serie completa (cumplimiento ~85% por construccion, con ejecutadas
nunca superior a planificadas). El cociente se calcula solo sobre iniciativas
que tienen ambos datos, sin fabricar ceros.

In [ ]:
def cumplimiento_por_anio(df):
    """I19 por anio: suma ejecutadas / suma planificadas, solo filas con ambos datos."""
    ambos = df.dropna(subset=["cantidad_act_planificadas", "cantidad_act_ejecutadas"])
    kpi = ambos.groupby("anio")[["cantidad_act_planificadas", "cantidad_act_ejecutadas"]].sum(min_count=1)
    return (100 * kpi["cantidad_act_ejecutadas"] / kpi["cantidad_act_planificadas"]).reindex(ANIOS)

fig, axr, axs = par("KPI I19: % cumplimiento (ejecutadas / planificadas) por anio")
barras_anio(axr, cumplimiento_por_anio(real), C_REAL,
            "Dataset REAL (ejecutadas se capturo solo 2022-2023)", pct=True)
barras_anio(axs, cumplimiento_por_anio(sint), C_SINT, ETIQ, pct=True)
for ax in (axr, axs):
    ax.set_ylim(0, 115)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

In [ ]:
# I19 por carrera y anio (solo sintetico): top 12 carreras por volumen
top_car = sint["carrera"].value_counts().head(12).index
sub = sint[sint["carrera"].isin(top_car)]
tab = sub.groupby(["carrera", "anio"]).apply(
    lambda g: 100 * g["cantidad_act_ejecutadas"].sum() / g["cantidad_act_planificadas"].sum()
).unstack().reindex(top_car)
tab.index = [c.replace("_", " ")[:32] for c in tab.index]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(tab, annot=True, fmt=".0f", cmap=SEQ_AZUL, vmin=60, vmax=100,
            linewidths=2, linecolor="white", cbar_kws={"label": "% cumplimiento"}, ax=ax)
ax.set_title(f"KPI I19 por carrera y anio (top 12 por volumen)  {ETIQ}",
             color=INK, fontsize=11, loc="left")
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(colors=MUTED, labelsize=9)
plt.tight_layout()
plt.show()

## Grafico especial 1 - Evolucion de la VcM en el tiempo

Lineas de numero de iniciativas por anio: total institucional y por facultad, y
abajo las 5 carreras con mas iniciativas. **Calculable con datos reales** en
todo el periodo (la evolucion por anio del sintetico es identica por
construccion). Cada facultad/carrera conserva su color en ambos paneles.

In [ ]:
fig, axr, axs = par("Evolucion de iniciativas por anio: institucional y por facultad", figsize=(12.5, 4.6))
for ax, df, tit in [(axr, real, "Dataset REAL"), (axs, sint, ETIQ)]:
    tot = df.groupby("anio").size().reindex(ANIOS)
    ax.plot(ANIOS, tot.values, color=INK, linewidth=2.4, marker="o",
            markersize=5, label="Institucional")
    tab = df.groupby(["anio", "facultad"]).size().unstack().reindex(
        index=ANIOS, columns=FACULTADES)
    for f in FACULTADES:
        ax.plot(ANIOS, tab[f].fillna(0).values, color=FCOLOR[f], linewidth=1.8,
                marker="o", markersize=4, label=f)
    ax.set_xticks(ANIOS)
    estilo(ax, tit)
axr.legend(frameon=False, fontsize=8, labelcolor=INK, ncols=2)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

In [ ]:
TOP5 = real["carrera"].value_counts().head(5).index.tolist()
CCOLOR = {c: PAL8[i] for i, c in enumerate(TOP5)}

fig, axr, axs = par("Evolucion por carrera (top 5 con mas iniciativas)", figsize=(12.5, 4.2))
for ax, df, tit in [(axr, real, "Dataset REAL"), (axs, sint, ETIQ)]:
    tab = df[df["carrera"].isin(TOP5)].groupby(["anio", "carrera"]).size().unstack()
    tab = tab.reindex(index=ANIOS, columns=TOP5)
    for c in TOP5:
        ax.plot(ANIOS, tab[c].fillna(0).values, color=CCOLOR[c], linewidth=1.8,
                marker="o", markersize=4, label=c.replace("_", " ")[:28])
    ax.set_xticks(ANIOS)
    estilo(ax, tit)
axr.legend(frameon=False, fontsize=8, labelcolor=INK)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## Grafico especial 2 - Actividades en el territorio de la RM

Ranking de comunas de la RM por numero de iniciativas, para ver concentraciones
y dispersiones territoriales. Requiere `comuna_rm`, que la fuente real no
captura: **solo demostrable con el sintetico** (donde la asignacion de comunas
es aleatoria uniforme, por eso el ranking demo se ve parejo; con datos reales
mostraria la concentracion efectiva).

In [ ]:
fig, axr, axs = par("Territorio RM: iniciativas por comuna", figsize=(12.5, 5.2))
sin_datos(axr, "La fuente no captura la comuna de ejecucion\n(comuna_rm vacia en todo el periodo)")
top = sint["comuna_rm"].value_counts()
mostrar = top.head(15).sort_values()
axs.barh(mostrar.index, mostrar.values.astype(float), height=0.6, color=C_SINT,
         edgecolor="white", linewidth=0.5)
axs.grid(axis="x", color=GRID, linewidth=0.8)
axs.grid(axis="y", visible=False)
for s in ("top", "right", "left"):
    axs.spines[s].set_visible(False)
axs.tick_params(colors=MUTED, labelsize=8.5)
axs.set_title(f"{ETIQ}: top 15 de {top.shape[0]} comunas con iniciativas",
              color=INK, fontsize=10, loc="left")
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## Cierre - Estado de la capacidad de reporteria con datos reales

Resumen por KPI: si es calculable **hoy** con el dataset real, y desde que anio.
"Parcial" significa que el dato existe pero no en todo el periodo o no con
cobertura completa.

| # | KPI | Calculable con datos reales | Desde | Nota |
|---|-----|------------------------------|-------|------|
| 1 | % dominios disciplinares en proceso formativo | Parcial | 2024 | Multivalor extraido de la fuente (~88% de cobertura); el % literal sobre el catalogo por carrera requiere ese catalogo |
| 2 | % iniciativas realizadas por anio | Si | 2022 | Derivado de estado_sisav normalizado |
| 3 | % iniciativas en comunas RM | No | - | La fuente no captura comuna |
| 4 | N iniciativas total y por facultad | Si | 2022 | |
| 5 | N iniciativas por instrumento y anio | Si | 2022 | |
| 6 | N estudiantes participantes | Parcial | 2024 | 2022-2023 sin desagregacion en la fuente |
| 7 | N academicos en iniciativas | Parcial | 2024 | Idem |
| 8 | N titulados (total / por rol) | Parcial / No | 2024 / - | Total desde 2024; roles no se capturan |
| 9 | N empleadores (por facultad y rol) | No | - | Sin columna en ninguna fuente 2022-2025 |
| 10 | N organizaciones OSC (total / por rol) | Parcial / No | 2024 / - | Total desde 2024; roles no se capturan |
| 11 | N competencias sello desagregadas | Si | 2022 | Flags derivados del texto real |
| 12 | N iniciativas con internacionalizacion | No | - | No se captura |
| 13 | N catedras declaradas | Parcial | 2024 | Cobertura ~57% (por instrumento) |
| 14 | N iniciativas por ciclo de estudio | No | - | No esta en el esquema v1 |
| 15 | % iniciativas con evidencia | Si | 2022 | SI/NO normalizado; 2022-2023 derivado de Estado de Evidencia |
| I19 | Contribucion al plan de estudios (ejec/planif) | Parcial | 2022 | Real solo 2022-2023: la fuente dejo de capturar ejecutadas desde 2024 |
| E1 | Evolucion VcM en el tiempo (lineas) | Si | 2022 | Institucional, facultad y carrera |
| E2 | Territorio RM por comuna | No | - | Requiere captura de comuna |

**Sintesis para VcM:** hoy la reporteria real cubre completa la actividad
(iniciativas, instrumentos, estados, evolucion), las competencias sello y la
evidencia de ejecucion en todo el periodo; desde 2024 los participantes
desagregados y los dominios disciplinares; y el I19 con datos verdaderos para
2022-2023 (recuperar el I19 hacia adelante requiere que la fuente vuelva a
capturar las actividades ejecutadas, que dejo de registrarse en 2024). Todo lo
territorial, los roles, empleadores, internacionalizacion y ciclo requieren que
el instrumento **empiece a capturar esos campos**; mientras tanto, el dashboard
se demuestra completo con el dataset sintetico, siempre etiquetado como
demostrativo.